# Hybrid Transformer vs. Standard Transformer Comparison

This notebook compares the standard Probabilistic Transformer against the Hybrid Transformer which integrates an Ornstein-Uhlenbeck (OU) process to model residuals

## Methodology

1.  Standard transformer: predicting prices solely based on input features
2.  Hybrid Transformer:
    - Train Transformer for trend prediction
    - Model residuals (Actual - Predicted) using an OU process
    - Final forecast = transformer trend + OU process mean reversion

We run both models 10 times to reduce statistical variability and report average metrics (MAE, RMSE, MAPE, R2, Pinball Loss)

In [1]:
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

current_dir = Path.cwd()
project_root = None
if (current_dir / 'config.py').exists():
    project_root = str(current_dir)
elif (current_dir.parent / 'config.py').exists():
    project_root = str(current_dir.parent)

if project_root and project_root not in sys.path:
    sys.path.insert(0, project_root)

from models import ProbabilisticTransformer, HybridProbabilisticTransformer
from core.experiment_utils import (
    load_data, load_cache, save_cache, run_experiment, N_RUNS,
)

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

try:
    gpus = tf.config.experimental.list_physical_devices("GPU")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPUs detected: {len(gpus)}")
except Exception as e:
    print(f"GPU config failed: {e}")

2026-03-07 21:17:15.006927: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772914635.099136 2006229 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772914635.124952 2006229 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772914635.370039 2006229 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772914635.370102 2006229 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772914635.370110 2006229 computation_placer.cc:177] computation placer alr

GPUs detected: 1


In [2]:
RESULTS_DIR = Path(project_root) / "results" / "hybrid_comparison"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_FILE = RESULTS_DIR / "results.json"

In [3]:
# All metrics and caching handled by core.experiment_utils

In [4]:
data = load_data()


Loading Data...
NaN remaining - Train: 0, Val: 0, Test: 0


In [5]:
cache = load_cache(CACHE_FILE)

run_experiment(
    ProbabilisticTransformer, "Standard Transformer", data,
    str(RESULTS_DIR), cache=cache, is_hybrid=False,
)

run_experiment(
    HybridProbabilisticTransformer, "Hybrid (Transformer+OU)", data,
    str(RESULTS_DIR), cache=cache, is_hybrid=True,
)

save_cache(CACHE_FILE, cache)

Results for Standard Transformer found in cache.
Results for Hybrid (Transformer+OU) found in cache.


In [6]:
results = [{"Model": k, **v} for k, v in cache.items()]
df_res = pd.DataFrame(results)
if not df_res.empty:
    display(df_res.sort_values("MAE"))
    df_res.to_csv(RESULTS_DIR / "comparison_summary.csv", index=False)

,Model,MAE,RMSE,MAPE,R2,Pinball_10,Pinball_50,Pinball_90,Avg_Pinball
1,Hybrid (Transformer+OU),24.911160,33.798185,863.539976,0.134018,5.862197,12.256008,6.877298,8.331835
0,Standard Transformer,25.496995,33.997641,839.510828,0.123444,6.090563,12.422647,7.229250,8.580820
